# Aula 03 — Regressão linear por OLS

Nesta aula, vamos ler a saída da Aula 2 e ajustar um modelo de regressão linear múltipla.

## Objetivos

- consumir a amostra saneada;
- escolher variáveis explicativas;
- ajustar o modelo OLS;
- gerar resíduos e salvar a saída para a Aula 4.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import resolve_project_root
from servicos.regressao import (
    attach_predictions_and_residuals,
    build_coefficients_table,
    build_model_summary,
    fit_ols_regression,
)

In [2]:
TARGET_COLUMN: Final[str] = 'preco'
PREFERRED_FEATURES: Final[tuple[str, ...]] = (
    'areaprivativa',
    'vagas',
    'distanciacentrokm',
    'dist_praia',
)


def locate_input_file(project_root: Path) -> Path:
    candidate = project_root / 'data' / 'output' / 'aula_02_amostra_saneada.csv'
    if candidate.exists():
        return candidate
    raise FileNotFoundError('Saída da Aula 2 não encontrada em data/output/.')

## Etapa 1 — Ler a saída da Aula 2

In [3]:
project_root = resolve_project_root()
input_path = locate_input_file(project_root)
df_model = pd.read_csv(input_path)
print('Base carregada da Aula 2:', df_model.shape)
df_model.head()

Base carregada da Aula 2: (17, 8)


,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001,8823.529412
1,AP-002,820000.0,92.5,2,8.0,1.8,https://portalimoveis.com.br/anuncio/002,8864.864865
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003,8846.153846
3,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005,8923.076923
4,AP-006,950000.0,105.0,2,4.0,1.5,https://portalimoveis.com.br/anuncio/006,9047.619048


## Etapa 2 — Selecionar variáveis

In [4]:
available_features = [column for column in PREFERRED_FEATURES if column in df_model.columns]
print('Variável dependente:', TARGET_COLUMN)
print('Variáveis explicativas disponíveis:', available_features)

Variável dependente: preco
Variáveis explicativas disponíveis: ['areaprivativa', 'vagas', 'distanciacentrokm']


## Etapa 3 — Ajustar o modelo

In [5]:
model = fit_ols_regression(
    df=df_model,
    target_col=TARGET_COLUMN,
    feature_columns=available_features,
)
summary = build_model_summary(model)
coeff_table = build_coefficients_table(model)
summary

{'n_obs': 17,
 'n_params': 4,
 'gl_modelo': 3,
 'gl_residuos': 13,
 'r2': 0.9971825954411132,
 'r2_ajustado': 0.9965324251582932,
 'rmse': 6332.975804797668,
 'sse': 681811903.2505956,
 'ssr': 241318188096.7494,
 'sst': 242000000000.0,
 'f_statistic': 1533.7252744249943,
 'f_p_value': 1.1102230246251565e-16}

## Etapa 4 — Ler a equação estimada

In [6]:
def resolve_equation_terms(coeff_table: pd.DataFrame) -> str:
    pieces: list[str] = []
    for _, row in coeff_table.iterrows():
        variable = str(row['variavel'])
        coefficient = float(row['coeficiente'])
        if variable == 'const':
            pieces.append(f'{coefficient:,.2f}')
            continue
        signal = '+' if coefficient >= 0 else '-'
        pieces.append(f'{signal} {abs(coefficient):,.2f}·{variable}')
    return 'Preço = ' + ' '.join(pieces)

print(resolve_equation_terms(coeff_table))
coeff_table

Preço = -51,937.89 + 9,555.44·areaprivativa - 2,530.68·vagas + 2,439.89·distanciacentrokm


,variavel,coeficiente,erro_padrao,estatistica_t,p_valor,significativo_10pct
0,const,-51937.888738,33119.885658,-1.568178,1.168395e-01,NAO
1,areaprivativa,9555.442004,372.185782,25.673850,2.290501e-145,SIM
2,vagas,-2530.681903,6702.507753,-0.377572,7.057483e-01,NAO
3,distanciacentrokm,2439.893480,3901.049563,0.625445,5.316788e-01,NAO


## Etapa 5 — Gerar resíduos e salvar saída

In [7]:
enriched_df = attach_predictions_and_residuals(df_model, model)
output_dir = project_root / 'data' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'aula_03_amostra_com_residuos.csv'
enriched_df.to_csv(output_path, index=False)
print(output_path)

/Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia/data/output/aula_03_amostra_com_residuos.csv


## Conclusão

A Aula 3 termina com a base modelada e salva. A Aula 4 vai usar essa saída.